# Modelling
## Setup

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import balanced_accuracy_score, f1_score, classification_report

df = pd.read_csv('features.csv')

# X bersih: tanpa id/label/confound
drop_cols = ['speaker_id', 'class']
feature_cols = [c for c in df.columns if c not in drop_cols]
X = df[feature_cols].values
y = df['class'].values
groups = df['speaker_id'].values

logo = LeaveOneGroupOut()

## Loop LOSO per classifier

In [2]:
def run_loso(make_model, name):
    y_true, y_pred, y_prob = [], [], []
    for train_idx, test_idx in logo.split(X, y, groups):
        Xtr, Xte = X[train_idx], X[test_idx]
        ytr = y[train_idx]

        # scaling WAJIB di dalam fold (hindari data leakage dari speaker test)
        scaler = StandardScaler().fit(Xtr)
        model = make_model()
        model.fit(scaler.transform(Xtr), ytr)

        y_true.append(y[test_idx[0]])          # 1 sample per fold = 1 speaker
        y_pred.append(model.predict(scaler.transform(Xte))[0])
        if hasattr(model, 'predict_proba'):
            proba = model.predict_proba(scaler.transform(Xte))[0]
        else:
            proba = model.decision_function(scaler.transform(Xte))
            proba = 1 / (1 + np.exp(-proba))   # squash ke [0,1] kira-kira
        # proba kelas 'native' = nativeness score
        native_idx = list(model.classes_).index('native')
        y_prob.append(proba[native_idx] if proba.ndim > 0 else proba)

    bal_acc = balanced_accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    print(f"=== {name} ===")
    print(f"Balanced Acc : {bal_acc:.2%}")
    print(f"Macro F1     : {macro_f1:.2%}")
    print(classification_report(y_true, y_pred, labels=['native', 'learner']))
    return np.array(y_true), np.array(y_pred), np.array(y_prob)

models = {
    'SVM (RBF)': lambda: SVC(kernel='rbf', class_weight='balanced',
                             probability=True, random_state=42),
    'Random Forest': lambda: RandomForestClassifier(
        n_estimators=300, class_weight='balanced', random_state=42),
    'kNN (k=5)': lambda: KNeighborsClassifier(n_neighbors=5),
}

results = {}
for name, maker in models.items():
    results[name] = run_loso(maker, name)

/home/kopiadem/Documents/MASTER/1. SEM-1/pengenalan-pola/assignment1/jp-native-vs-learner/.venv/lib64/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/kopiadem/Documents/MASTER/1. SEM-1/pengenalan-pola/assignment1/jp-native-vs-learner/.venv/lib64/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/kopiadem/Documents/MASTER/1. SEM-1/pengenalan-pola/assignment1/jp-native-vs-learner/.venv/lib64/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV

=== SVM (RBF) ===
Balanced Acc : 50.00%
Macro F1     : 50.00%
              precision    recall  f1-score   support

      native       0.50      0.50      0.50         6
     learner       0.50      0.50      0.50         6

    accuracy                           0.50        12
   macro avg       0.50      0.50      0.50        12
weighted avg       0.50      0.50      0.50        12

=== Random Forest ===
Balanced Acc : 66.67%
Macro F1     : 66.67%
              precision    recall  f1-score   support

      native       0.67      0.67      0.67         6
     learner       0.67      0.67      0.67         6

    accuracy                           0.67        12
   macro avg       0.67      0.67      0.67        12
weighted avg       0.67      0.67      0.67        12

=== kNN (k=5) ===
Balanced Acc : 66.67%
Macro F1     : 65.71%
              precision    recall  f1-score   support

      native       0.75      0.50      0.60         6
     learner       0.62      0.83      0.71    

## Native Sccore per speaker

In [3]:
for name, (y_true, y_pred, y_prob) in results.items():
    score_df = pd.DataFrame({
        'speaker_id': np.unique(groups),
        'true': y_true,
        'pred': y_pred,
        'nativeness_score': y_prob,
    }).merge(df[['speaker_id']].drop_duplicates()
             .assign(speaker_id=lambda d: d['speaker_id']), on='speaker_id')
    # tambah gender/location dari metadata utk analisis
    score_df = score_df.merge(
        pd.read_csv('metadata_utterance.csv')[['speaker_id', 'gender', 'location']]
        .drop_duplicates('speaker_id'), on='speaker_id')
    score_df.to_csv(f'nativeness_{name.split()[0].lower()}.csv', index=False)
    print(f"\n--- {name}: nativeness score per speaker ---")
    print(score_df.sort_values('nativeness_score')
          .to_string(index=False))


--- SVM (RBF): nativeness score per speaker ---
speaker_id    true    pred  nativeness_score gender location
 learner01 learner learner          0.145501      M   indoor
 learner03 learner learner          0.358731      M   indoor
  native06  native learner          0.391436      M   indoor
 learner02 learner learner          0.455634      M   indoor
  native04  native learner          0.471309      F     cafe
  native01  native learner          0.479181      M     cafe
 learner06 learner  native          0.500000      M   indoor
  native05  native  native          0.566112      F     cafe
  native02  native  native          0.787902      M     cafe
  native03  native  native          0.788850      F     cafe
 learner04 learner  native          0.809120      M   indoor
 learner05 learner  native          0.966984      F   indoor

--- Random Forest: nativeness score per speaker ---
speaker_id    true    pred  nativeness_score gender location
 learner01 learner learner          0.293333

## Gender Stratified

In [4]:
best_name = max(results, key=lambda n: balanced_accuracy_score(*results[n][:2]))
y_true, y_pred, y_prob = results[best_name]
chk = pd.DataFrame({'speaker_id': np.unique(groups),
                    'true': y_true, 'pred': y_pred,
                    'score': y_prob}).merge(
    pd.read_csv('metadata_utterance.csv')[['speaker_id', 'gender']]
    .drop_duplicates('speaker_id'), on='speaker_id')

male = chk[chk['gender'] == 'M']
print(f"Male speakers  : {male['pred'].eq(male['true']).mean():.2%} benar "
      f"({male['pred'].eq(male['true']).sum()}/{len(male)})")
print("\nF0 std rata-rata per kelas (dari features.csv):")
print(df.groupby('class')['f0_std'].agg(['mean', 'std']))

Male speakers  : 87.50% benar (7/8)

F0 std rata-rata per kelas (dari features.csv):
              mean        std
class                        
learner  51.701742  22.706527
native   50.031216  19.216867
